### **Extraer el `nombre`, `precio` y `enlace` de los anuncios en la pagina de Mercado libre**

Vamos a scrapear el siguiente sitio:

https://listado.mercadolibre.cl/herramientas-vehiculos

Vamos a inspeccionar el sitio y ubicaremos primero el boton de '`Aceptar cokies`' para darle click y no nos genere algun tipo de problema:

<center><img src="https://i.postimg.cc/SxjLB19L/ws-319.png"></center>
<center><img src="https://i.postimg.cc/PrVYQb7C/ws-320.png"></center>

Luego, busco el boton para ir a la `página 10`, dado que al pulsar esta página se carga el número de la última página en la barra de paginación:

<center><img src="https://i.postimg.cc/rwbGMGGB/ws-321.png"></center>
<center><img src="https://i.postimg.cc/TY3nbNt8/ws-323.png"></center>
<center><img src="https://i.postimg.cc/X7c4M3FF/ws-331.png"></center>

Localizamos la barra de paginación con el fin de extraer el número de la última página

<center><img src="https://i.postimg.cc/NFgmpdYH/ws-324.png"></center>
<center><img src="https://i.postimg.cc/7ZpzpHwD/ws-325.png"></center>

Estando en la página 10, busco el boton para ir a la `página 1` para regresar a la página inicial

<center><img src="https://i.postimg.cc/Tw4hDxQh/ws-332.png"></center>

Localizar la caja que contiene todos los articulos listados en la página:

<center><img src="https://i.postimg.cc/6p7Rj2sY/ws-326.png"></center>

Obtener todos los articulos listados

<center><img src="https://i.postimg.cc/wBLNzCjN/ws-327.png"></center>

Buscamos el tag que contenga el `titulo` de cada articulo:

<center><img src="https://i.postimg.cc/B652TrW3/ws-328.png"></center>

Buscamos el tag que contenga el `precio` de cada articulo:

<center><img src="https://i.postimg.cc/63KvpM8m/ws-329.png"></center>

Buscamos el tag que contenga el `enlace` de cada articulo:

<center><img src="https://i.postimg.cc/vZTnDPGP/ws-333.png"></center>

Por tanto, escribimos el siguiente código:

In [1]:
import random
from time import sleep
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
#from webdriver_manager.chrome import ChromeDriverManager

# Definimos el User Agent en Selenium utilizando la clase Options
options = Options()
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

#URL SEMILLA
website = "https://listado.mercadolibre.cl/herramientas-vehiculos/"
# Se recomienda utilizar Chrome, pero podriamos utilizar Firefox, Safari, Edge, etc.
# driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()), options=options)
driver = webdriver.Chrome(options=options)
driver.get(website)
driver.maximize_window()

sleep(3) # Esperar a que todo cargue correctamente

# Debemos darle click al boton de 'disclaimer' para que no interrumpa nuestras acciones
# El boton disclaimer corresponde al boton de 'Aceptar cokies' 'Configurar cookies'
try: # Encerramos todo en un try catch para que si no aparece el discilamer, no se caiga el codigo
  disclaimer = driver.find_element(By.XPATH, '//button[@data-testid="action:understood-button"]')
  disclaimer.click() #  Damos click en el boton
except Exception as e:
  print (e) 

# Busco el boton para ir a la página 10, dado que al pulsar esta página se carga el número de la última página en 
# la barra de paginación
boton = driver.find_element(By.XPATH, '//button[@aria-label="Ir a la página 10"]')
boton.click()
# Espero que cargue la informacion dinamica
# Devuelve un número aleatorio entre, e incluido, 8 y 10:
sleep(random.uniform(3.0, 4.0))

# Paginación 1
# Localización de la barra de paginación
paginacion = driver.find_element(by=By.XPATH, value='//ul[contains(@class, "andes-pagination")]')  
# Localizar cada página mostrada en la barra de paginación
paginas = paginacion.find_elements(by=By.TAG_NAME, value='li') 
# Obtener la última página con indexación negativa (comienza desde donde termina el array)
ultima_pagina = int(paginas[-2].text) 

# Devolverme a la página 1
boton_pagina_inicial = driver.find_element(by=By.XPATH, value='//button[@aria-label="Ir a la página 1"]')
boton_pagina_inicial.click()
sleep(4)

# Inicializando el almacenamiento
articulo_titulo = []
articulo_precio = []
articulo_enlace = []

# Paginación 2
# Esta es la página que el bot comienza a scrapear
pagina_actual = 1  

# El bucle while funcionará hasta que el bot llegue a la última página del sitio web, entonces se interrumpirá (break)
while pagina_actual <= ultima_pagina:

    sleep(2)  # Dejar que la página se muestre correctamente
    # Localizar la caja que contiene todos los articulos listados en la página
    container = driver.find_element(by=By.XPATH, value='//ol[@class="ui-search-layout ui-search-layout--stack shops__layout"]')
    # Obtener todos los articulos listados (el "/" da nodos hijos inmediatos)
    articulos = container.find_elements(by=By.XPATH, value='.//div[@class="ui-search-result__content-wrapper"]')

    # Recorrer la lista de libros (cada "libro" es un audiolibro)
    for articulo in articulos:
        # Usamos "contains" para buscar elementos web que contengan un texto concreto, así evitamos construir XPATH largos
        articulo_titulo.append(articulo.find_element(by=By.XPATH, value='.//h2[contains(@class, "ui-search-item__title")]').text)  # Almacenar los datos en una lista
        articulo_precio.append(articulo.find_element(by=By.XPATH, value='.//span[contains(@class, "ui-search-price__part--medium")]/span[@class="andes-money-amount__fraction"]').text)
        articulo_enlace.append(articulo.find_element(by=By.XPATH, value='.//a[contains(@class, "ui-search-item__group__element")]').get_attribute("href"))

    # Incrementar la página_actual en 1 después de extraer los datos
    pagina_actual = pagina_actual + 1 
    # Localizar el botón 'siguiente_pagina' y pulsar sobre él. Si el elemento no está en la página web, pasar a la siguiente iteración.
    try:
        # siguiente_pagina = driver.find_element(by=By.XPATH, value='//span[text()="Siguiente"]')
        siguiente_pagina = driver.find_element(by=By.XPATH, value='//a[@title="Siguiente"]')
        siguiente_pagina.click()
    except:
        pass

driver.quit()

# Almacenar los datos en un DataFrame y exportar a un archivo CSV
df_articulos = pd.DataFrame({'titulo': articulo_titulo, 'precio': articulo_precio, 'enlace':articulo_enlace})
df_articulos.to_csv('articulos.csv', index=False)